In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
DATA_PATH = Path("data/7817_1.csv")

df = pd.read_csv(DATA_PATH)
df.sample(5)

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
184,AV1T09fyvKc47QAVgf2R,B00NO8LX7E,Amazon,Amazon Devices,NaN,2017-07-18T03:52:58Z,2017-08-13T08:28:47Z,NaN,NaN,amazonfiretvgamecontroller/b00no8lx7e,...,5.0,https://www.amazon.com/Amazon-Fire-TV-Game-Con...,We are HUGE fans of the Amazon Fire Stick and ...,Works great though and I know I can return if ...,NaN,NaN,Ashley,NaN,NaN,NaN
587,AVpjWh8e1cnluZ0-Vy0O,B00LWHU9D8,Amazon,"Electronics,Amazon Devices",GB,2016-05-20T11:07:30Z,2017-07-25T17:42:39Z,NaN,NaN,firehd6tablet/b00lwhu9d8,...,NaN,http://www.amazon.com/Fire-HD-Display-Wi-Fi-GB...,"For the low price, this tablet really does mor...","HD Fire 6 is great for the price! 3,544 people...",NaN,NaN,Earthling1984,NaN,NaN,NaN
1521,AVpge-anilAPnD_xtDVf,B00HX0SRXW,Amazon,"Amazon Devices,Corded Headsets,Electronics Fea...",Black,2015-07-13T14:50:00Z,2017-08-13T08:29:03Z,NaN,8.487190e+11,"amazonpremiumheadphones/b00hx0srxw,08487190395...",...,NaN,http://www.amazon.com/Amazon-KA416Y-Premium-He...,While I've purchased items from Amazon for yea...,Awesome Headphones! Longevity An Issue 733 of ...,NaN,NaN,A. Younan,NaN,8.487190e+11,NaN
488,AVzRloqLGV-KLJ3aavBd,B07194GPJV,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:52Z,NaN,NaN,allnewamazonfire7tabletcase7thgeneration/b0719...,...,5.0,https://www.amazon.com/All-New-Amazon-Tablet-G...,Fits greatly with the device and easy to use. ...,Accessories that complment my device,NaN,NaN,Amazon Customer,NaN,NaN,NaN
1071,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,3.0,http://reviews.bestbuy.com/3545/5097300/review...,"Great sound, it needs to be a little more soph...",A little hard to teach it,NaN,NaN,Denverbrian,NaN,8.416670e+11,1.75 lbs


In [13]:
df.isnull().sum()

id                         0
asins                      0
brand                      0
categories                 0
colors                   823
dateAdded                  0
dateUpdated                0
dimension               1032
ean                      699
keys                       0
manufacturer             632
manufacturerNumber       695
name                       0
prices                     0
reviews.date             380
reviews.doRecommend     1058
reviews.numHelpful       697
reviews.rating           420
reviews.sourceURLs         0
reviews.text               0
reviews.title             17
reviews.userCity        1597
reviews.userProvince    1597
reviews.username          17
sizes                   1597
upc                      699
weight                   911
aspect_cluster             0
aspect                     0
label                      0
dtype: int64


### Label Creation (Supervised Signal)

In [8]:
def label_sentiment(rating):
    return 1 if rating >= 4 else 0

df["label"] = df["reviews.rating"].apply(label_sentiment)

### Text Cleaning (Implicit)

In [9]:
df["reviews.text"] = df["reviews.text"].astype(str).str.strip()


### TF-IDF Vectorization

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

MAX_FEATURES = 5000

tfidf = TfidfVectorizer(
    max_features=MAX_FEATURES,
    stop_words="english",
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df["reviews.text"])

### KMeans Clustering (Aspect Extraction)

In [11]:
from sklearn.cluster import KMeans

NUM_CLUSTERS = 5  # You can tune this (4–8 ideal)

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
kmeans.fit(X)

df["aspect_cluster"] = kmeans.labels_

In [12]:
df.sample(5)

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight,aspect_cluster,aspect,label
42,AVpfBEWcilAPnD_xTGb7,B002Y27P3M,Amazon,"Kindle Store,Amazon Devices,Electronics",NaN,2015-01-17T12:14:51Z,2015-11-01T00:05:54Z,NaN,NaN,"kindlekeyboard/b002y27p3m,amazon/d01101",...,A hesistant buyer rejoices on his choice,NaN,NaN,Mr Goodwrench,NaN,NaN,1.1 pounds,2,price,0
1446,AVpge-anilAPnD_xtDVf,B00HX0SRXW,Amazon,"Amazon Devices,Corded Headsets,Electronics Fea...",Black,2015-07-13T14:50:00Z,2017-08-13T08:29:03Z,NaN,8.487190e+11,"amazonpremiumheadphones/b00hx0srxw,08487190395...",...,I hate having to shove headphones into my brai...,NaN,NaN,Andrew,NaN,8.487190e+11,NaN,4,usability,0
879,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,Not impressed,NaN,NaN,Pogera,NaN,8.416670e+11,1.75 lbs,2,price,0
688,AVzRlo37glJLPUi8FbPy,B01LW1MS9C,Amazon,"Amazon Devices,Kindle Store",NaN,2017-06-22T20:55:23Z,2017-08-13T08:28:46Z,NaN,NaN,amazonechodotcasefitsechodot2ndgenerationonlyc...,...,Great look for Dot.,NaN,NaN,C. Morrow,NaN,NaN,NaN,3,packaging,1
890,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,Dissatisfied,NaN,NaN,Jvier,NaN,8.416670e+11,1.75 lbs,3,packaging,0


### Extract Top Words per Cluster
Finds most important words in each cluster
Each cluster has a center vector

We:

Sort words by importance
Pick top N words

In [14]:
import numpy as np

def get_top_words_per_cluster(tfidf, model, n_words=10):
    words = tfidf.get_feature_names_out()
    clusters = {}

    for i in range(model.n_clusters):
        center = model.cluster_centers_[i]
        top_indices = center.argsort()[-n_words:][::-1]
        top_words = [words[j] for j in top_indices]
        clusters[i] = top_words

    return clusters

cluster_keywords = get_top_words_per_cluster(tfidf, kmeans)

for cluster, words in cluster_keywords.items():
    print(f"\nCluster {cluster}:")
    print(words)


Cluster 0:
['ipad', 'kindle', 'year', 'hd', 'early reviews', 'tablet', 'early', 'reader', 'hdx', 'reviews']

Cluster 1:
['apple buds', 'buds', 'headphones', 'range', 'sound', 'apple', 've', 'reviews', 'don', 'people']

Cluster 2:
['kindle', 'tv', 'amazon', 'prime', 'screen', 'tablet', 'read', 'like', 'device', 'case']

Cluster 3:
['echo', 'tap', 'great', 'speaker', 'alexa', 'love', 'sound', 'music', 'use', 'good']

Cluster 4:
['headphones', 'earbuds', 'magnets', 'ears', 'like', 'noise', 'tangle', 'set', 'designed', 'set headphones']


### Assign Human-Readable Aspect Names

In [35]:
# Auto-label each cluster using its top keywords (no manual mapping).
aspect_map = {
    cluster_id: " / ".join(words[:3])
    for cluster_id, words in cluster_keywords.items()
}

df["aspect"] = df["aspect_cluster"].map(aspect_map)

aspect_map

{0: 'ipad / kindle / year',
 1: 'apple buds / buds / headphones',
 2: 'kindle / tv / amazon',
 3: 'echo / tap / great',
 4: 'headphones / earbuds / magnets'}

### Combine with Sentiment (YOUR DistilBERT)

In [36]:
df["sentiment"] = df["label"].map({0: "negative", 1: "positive"})

### Aggregate Insights

In [37]:
aspect_sentiment = df.groupby(["aspect", "sentiment"]).size().unstack().fillna(0)

In [38]:
aspect_sentiment

sentiment,negative,positive
aspect,,
apple buds / buds / headphones,59.0,0.0
echo / tap / great,53.0,551.0
headphones / earbuds / magnets,35.0,39.0
ipad / kindle / year,32.0,24.0
kindle / tv / amazon,441.0,363.0
